# 04 Credit Scoring

This notebook demonstrates the end-to-end credit limit scoring workflow:

- Load processed datasets and credit limit labels
- Build selected modeling features without data leakage
- Train baseline regressors and a LightGBM regressor
- Tune the best model with time-series cross-validation
- Evaluate regression performance on the test set
- Validate hard regulatory credit constraints
- Save the best credit scoring model in joblib format


In [ ]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
SRC_DIR = PROJECT_ROOT / 'src'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'src' / 'models'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from features import CreditDataPreprocessor, CreditFeatureSelector
from models import CreditModelEvaluator, CreditScorer


In [ ]:
# Configuration
TARGET_COLUMN = 'calibrated_credit_limit'
RISK_COLUMN = 'preloan_risk_label'
USE_SUBSET_FOR_DEMO = True
TRAIN_ROWS = 120000
VAL_ROWS = 30000
TEST_ROWS = 30000

# Set USE_SUBSET_FOR_DEMO to False for a full-scale run.
print({
    'TARGET_COLUMN': TARGET_COLUMN,
    'RISK_COLUMN': RISK_COLUMN,
    'USE_SUBSET_FOR_DEMO': USE_SUBSET_FOR_DEMO,
    'TRAIN_ROWS': TRAIN_ROWS,
    'VAL_ROWS': VAL_ROWS,
    'TEST_ROWS': TEST_ROWS,
})


In [ ]:
# Load train/validation/test data
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv', low_memory=False)
val_df = pd.read_csv(PROCESSED_DIR / 'val.csv', low_memory=False)
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv', low_memory=False)

if USE_SUBSET_FOR_DEMO:
    train_df = train_df.iloc[: min(TRAIN_ROWS, len(train_df))].copy()
    val_df = val_df.iloc[: min(VAL_ROWS, len(val_df))].copy()
    test_df = test_df.iloc[: min(TEST_ROWS, len(test_df))].copy()

print('train_df shape:', train_df.shape)
print('val_df shape:', val_df.shape)
print('test_df shape:', test_df.shape)
display(train_df.head())


In [ ]:
# Prepare regression targets and risk labels
y_train = train_df[TARGET_COLUMN].astype(float).copy()
y_val = val_df[TARGET_COLUMN].astype(float).copy()
y_test = test_df[TARGET_COLUMN].astype(float).copy()

risk_val = val_df[RISK_COLUMN].astype(int).copy()
risk_test = test_df[RISK_COLUMN].astype(int).copy()

print('Target summary in train:')
display(y_train.describe())


In [ ]:
# Build leak-free selected features for regression
preprocessor = CreditDataPreprocessor(target_column=RISK_COLUMN)
X_train_processed = preprocessor.fit_transform(train_df)
X_val_processed = preprocessor.transform(val_df)
X_test_processed = preprocessor.transform(test_df)

selector = CreditFeatureSelector(
    variance_threshold=0.01,
    correlation_threshold=0.8,
    top_k_features=50,
    use_pca=False,
)
X_train_selected = selector.fit_transform(X_train_processed, train_df[RISK_COLUMN].astype(int))
X_val_selected = selector.transform(X_val_processed)
X_test_selected = selector.transform(X_test_processed)

print('Selected feature shapes:')
print('X_train_selected:', X_train_selected.shape)
print('X_val_selected:', X_val_selected.shape)
print('X_test_selected:', X_test_selected.shape)
display(X_train_selected.head())


In [ ]:
# Train baseline regressors and the core LightGBM model
model_types = ['linear_regression', 'ridge', 'lasso', 'lightgbm']
model_results = []
trained_models = {}

for model_type in model_types:
    scorer = CreditScorer(model_type=model_type)
    scorer.fit(X_train_selected, y_train)
    val_pred = scorer.predict(X_val_selected)
    rmse = float(np.sqrt(mean_squared_error(y_val, val_pred)))
    mae = float(mean_absolute_error(y_val, val_pred))
    r2 = float(r2_score(y_val, val_pred))
    model_results.append({
        'model_type': model_type,
        'val_rmse': rmse,
        'val_mae': mae,
        'val_r2': r2,
    })
    trained_models[model_type] = scorer

model_result_df = pd.DataFrame(model_results).sort_values('val_rmse', ascending=True)
display(model_result_df)

best_model_type = model_result_df.iloc[0]['model_type']
print('Best validation model:', best_model_type)


In [ ]:
# Hyperparameter tuning on the best model
best_scorer = CreditScorer(model_type=best_model_type)
best_scorer.hyperparameter_tune(
    X_train_selected,
    y_train,
    X_val_selected,
    y_val,
)

tuning_summary = best_scorer.get_tuning_summary()
display(pd.DataFrame([tuning_summary.__dict__]))


In [ ]:
# Final regression evaluation on the test set
regression_evaluator = CreditModelEvaluator(
    model=best_scorer,
    X_test=X_test_selected,
    y_test=y_test,
)
regression_results = regression_evaluator.evaluate_regression()
print('Regression results:')
print(json.dumps(regression_results, indent=2, ensure_ascii=False))


In [ ]:
# Test hard regulatory constraints
X_test_for_constraints = X_test_selected.copy()
X_test_for_constraints['annual_inc'] = pd.to_numeric(test_df['annual_inc'], errors='coerce').fillna(0.0).to_numpy()
X_test_for_constraints['dti'] = pd.to_numeric(test_df['dti'], errors='coerce').fillna(0.0).to_numpy()

raw_predictions = best_scorer.predict(X_test_selected)
constrained_predictions = best_scorer.predict_with_constraints(
    X_test_for_constraints,
    risk_test,
)

constraint_check = pd.DataFrame({
    'risk_label': risk_test.to_numpy(),
    'annual_inc': X_test_for_constraints['annual_inc'].to_numpy(),
    'dti': X_test_for_constraints['dti'].to_numpy(),
    'raw_prediction': raw_predictions,
    'constrained_prediction': constrained_predictions,
})
constraint_check['monthly_disposable_income'] = (
    constraint_check['annual_inc'] * (1 - constraint_check['dti'].clip(lower=0, upper=100) / 100.0) / 12.0
).clip(lower=0.0)
constraint_check['regulatory_cap'] = np.select(
    [
        constraint_check['risk_label'].isin([3, 4]),
        constraint_check['risk_label'].eq(2),
        constraint_check['risk_label'].eq(1),
        constraint_check['risk_label'].eq(0),
    ],
    [
        0.0,
        constraint_check['monthly_disposable_income'] * 3.0,
        constraint_check['monthly_disposable_income'] * 6.0,
        constraint_check['monthly_disposable_income'] * 12.0,
    ],
    default=0.0,
)

print('Constraint validation summary:')
print('Violations after constraints:', int((constraint_check['constrained_prediction'] > constraint_check['regulatory_cap'] + 1e-6).sum()))
display(constraint_check.head(20))


In [ ]:
# Save the best tuned scorer
model_output_path = MODEL_DIR / f'best_{best_model_type}_credit_scorer.joblib'
joblib.dump(best_scorer.best_model_, model_output_path)

metrics_output_path = MODEL_DIR / 'best_credit_scoring_metrics.json'
metrics_output_path.write_text(
    json.dumps(regression_results, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print('Saved credit scorer to:', model_output_path)
print('Saved regression metrics to:', metrics_output_path)
